# Generate data for dysts examples

First we load some packages:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dysts import flows
from dysts.systems import get_system_data
import h5py

Then, we simulate the chaotic systems, skipping those that have unbounded indices or are not autonomous.

In [ ]:
attractors = get_system_data("continuous_no_delay")
nPeriods = 10
counter = 0
for system_name in attractors:
    counter += 1
    unbounded_indices = attractors[system_name]["unbounded_indices"]
    nonautomonous = attractors[system_name]["nonautonomous"]

    if ( (nonautomonous) or (unbounded_indices) ):
        print("Skipping ", system_name, "(", counter, "/", len(attractors), ")")
    
    else:

        ## Display
        print("Simulating ", system_name, "(", counter, "/", len(attractors), ")")

        ## Get simulation parameters for desired timestep (20 * accurate one)
        pts_per_period = 1
        dt_sim = attractors[system_name]["period"]/pts_per_period
        dt_min = np.minimum(0.25, 20*attractors[system_name]["dt"])
        while (pts_per_period < 500) & (dt_sim > dt_min):
            pts_per_period += 1
            dt_sim = attractors[system_name]["period"]/pts_per_period
        
        ## Simulate
        model = getattr(flows, system_name)()
        nPoints = nPeriods * pts_per_period
        t, x = model.make_trajectory(nPoints, resample=True, return_times=True, pts_per_period=pts_per_period)
        
        ## Evaluate the vector field at the simulation points
        y = np.array([model(x[i], t[i]) for i in range(x.shape[0])])
        
        ## Save data
        filename = "./data-raw/" + system_name + ".hdf5"
        f = h5py.File(filename, "w")
        dt = np.mean( np.diff(t) )
        f.create_dataset("dt", data=dt)
        f.create_dataset("t" , data=t)
        f.create_dataset("x" , data=x)
        f.create_dataset("f" , data=y)
        f.create_dataset("periods" , data=nPeriods)
        f.create_dataset("pts_per_period", data=pts_per_period)
        f.close()